In [1]:
from dataloader.didemo_loader import DiDeMoDataset
from torch.utils.data import DataLoader

def collate_fn(batch):
    import torch
    from torch.nn.utils.rnn import pad_sequence

    video_feat = torch.stack([b["video_feat"] for b in batch])

    query_feat = pad_sequence(
        [b["query_feat"] for b in batch],
        batch_first=True
    )

    caption_feat = pad_sequence(
        [b["caption_feat"] for b in batch],
        batch_first=True
    )

    start = torch.tensor([b["start"] for b in batch], dtype=torch.long)
    end   = torch.tensor([b["end"] for b in batch], dtype=torch.long)

    video_id = [b["video_id"] for b in batch]

    return {
        "video_feat": video_feat,
        "query_feat": query_feat,
        "caption_feat": caption_feat,
        "start": start,
        "end": end,
        "video_id": video_id
    }

train_dataset = DiDeMoDataset("train")
val_dataset   = DiDeMoDataset("val")
test_dataset  = DiDeMoDataset("test")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)
sample = next(iter(train_loader))

print(sample["video_feat"].shape)
print(sample["query_feat"].shape)
print(sample["caption_feat"].shape)

c:\Users\Admin\anaconda3\envs\ml_env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 10368.65it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params 

torch.Size([32, 6, 384])
torch.Size([32, 14, 384])
torch.Size([32, 59, 384])


In [5]:
# ==========================================
# 🔷 FULL TRAINING PIPELINE- GPU
# ==========================================

import os
import torch
from tqdm import tqdm
import torch.nn.functional as F

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention
from model.bid_attn import BiDAttention
from model.region_predictor import RegionPredictor
from model.region_embedding import RegionEmbedding
from model.bigru import BiGRULocalizer

from losses.sc_loss import SemanticContrastiveLoss
from losses.ra_loss import RegionAwareLoss
from losses.pml_loss import MomentLocalizationLoss

# -------------------------------
# Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Hyperparameters
# -------------------------------
EPOCHS = 30
LR = 5e-5
PATIENCE = 10

W_ML = 1.0
W_SC = 0.01
W_RA = 0.005

# -------------------------------
# Initialize Model Components
# -------------------------------
query_split = QuerySplit(384).to(device)
mmt = MMT(384, 4, 1536, 1).to(device)
cross_attn = CrossAttention(384).to(device)
bid_attn = BiDAttention(384, 4).to(device)
region_pred = RegionPredictor(384).to(device)
region_embed = RegionEmbedding(384).to(device)
bigru = BiGRULocalizer().to(device)

modules = [query_split, mmt, cross_attn, bid_attn, region_pred, region_embed, bigru]

for m in modules:
    m.train()

# -------------------------------
# Losses
# -------------------------------
sc_loss_fn = SemanticContrastiveLoss()
ra_loss_fn = RegionAwareLoss()
ml_loss_fn = MomentLocalizationLoss()

# -------------------------------
# Optimizer
# -------------------------------
params = []
for m in modules:
    params += list(m.parameters())

optimizer = torch.optim.Adamax(params, lr=LR)

# -------------------------------
# Checkpoint dir
# -------------------------------
os.makedirs("checkpoints", exist_ok=True)

# -------------------------------
# Utility: IoU
# -------------------------------
def compute_iou(pred_s, pred_e, gt_s, gt_e):
    inter = max(0, min(pred_e, gt_e) - max(pred_s, gt_s) + 1)
    union = max(pred_e, gt_e) - min(pred_s, gt_s) + 1
    return inter / union if union > 0 else 0

# -------------------------------
# Validation
# -------------------------------
def evaluate():
    for m in modules:
        m.eval()

    total = 0
    correct_05 = 0
    correct_07 = 0

    with torch.no_grad():
        for batch in val_loader:

            video_feat = batch["video_feat"].to(device)
            query_feat = batch["query_feat"].to(device)
            caption_feat = batch["caption_feat"].to(device)

            start_gt = batch["start"].to(device)
            end_gt = batch["end"].to(device)

            N = video_feat.shape[1]

            Q_f, Q_s = query_split(query_feat)
            F_context = mmt(video_feat)
            S_context = mmt(caption_feat)

            F_prime = cross_attn(F_context, Q_f)
            S_prime = cross_attn(S_context, Q_s)

            C = bid_attn(F_prime, S_prime, query_feat)

            P_reg, _ = region_pred(C, use_gumbel=False)
            C_prime = region_embed(C, P_reg)

            start_logits, end_logits = bigru(C_prime)
            # 🔥 Sharpen predictions
            logit_temp = 0.3
            start_logits = start_logits / logit_temp
            end_logits   = end_logits / logit_temp
            
            start_logits = start_logits[:, :N]
            end_logits = end_logits[:, :N]

            pred_s = torch.argmax(start_logits, dim=1)
            pred_e = torch.argmax(end_logits, dim=1)

            for i in range(len(pred_s)):
                iou = compute_iou(pred_s[i].item(), pred_e[i].item(),
                                  start_gt[i].item(), end_gt[i].item())

                if iou >= 0.5:
                    correct_05 += 1
                if iou >= 0.7:
                    correct_07 += 1

                total += 1

    for m in modules:
        m.train()

    return correct_05 / total, correct_07 / total

# -------------------------------
# Training Loop
# -------------------------------
best_score = 0
no_improve = 0

for epoch in range(EPOCHS):

    epoch_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, batch in enumerate(pbar):

        video_feat = batch["video_feat"].to(device)
        query_feat = batch["query_feat"].to(device)
        caption_feat = batch["caption_feat"].to(device)

        start_gt = batch["start"].to(device)
        end_gt = batch["end"].to(device)

        B, N, _ = video_feat.shape

        optimizer.zero_grad()

        # -------------------------------
        # Forward pass
        # -------------------------------
        Q_f, Q_s = query_split(query_feat)

        F_context = mmt(video_feat)
        S_context = mmt(caption_feat)

        F_prime = cross_attn(F_context, Q_f)
        S_prime = cross_attn(S_context, Q_s)

        C = bid_attn(F_prime, S_prime, query_feat)

        # Gumbel Softmax (training)
        P_reg, logits = region_pred(C, use_gumbel=True)

        C_prime = region_embed(C, P_reg)

        start_logits, end_logits = bigru(C_prime)
        # 🔥 Sharpen predictions
        logit_temp = 0.3
        start_logits = start_logits / logit_temp
        end_logits   = end_logits / logit_temp
        # -------------------------------
        # L_ml
        # -------------------------------
        L_ml, _, _ = ml_loss_fn(start_logits, end_logits, start_gt, end_gt, N)

        # -------------------------------
        # L_sc (negative sampling)
        # -------------------------------
       
        F_pos = F_context
        S_pos = S_context

        with torch.no_grad():

            # ---- Query representation ----
            Q_global = query_feat.mean(dim=1)                      # (B, 384)
            Q_global = F.normalize(Q_global, dim=-1)

            # ---- Feature representation (global) ----
            F_global = F_pos.mean(dim=1)                           # (B, 384)
            S_global = S_pos.mean(dim=1)

            F_global = F.normalize(F_global, dim=-1)
            S_global = F.normalize(S_global, dim=-1)

            # ---- Similarity matrix ----
            sim_f = torch.matmul(Q_global, F_global.T)             # (B, B)
            sim_s = torch.matmul(Q_global, S_global.T)

            # ---- Mask self similarity ----
            mask = torch.eye(B, device=device).bool()
            sim_f.masked_fill_(mask, -1e9)
            sim_s.masked_fill_(mask, -1e9)

            # ---- Pick hardest negatives ----
            hard_idx_f = torch.argmax(sim_f, dim=1)                # (B,)
            hard_idx_s = torch.argmax(sim_s, dim=1)

        # ---- Gather negatives ----
        F_neg = F_pos[hard_idx_f]
        S_neg = S_pos[hard_idx_s]

        # ---- Compute loss ----
        L_sc = sc_loss_fn(query_feat, F_pos, F_neg, S_pos, S_neg)

       
        # -------------------------------
        # L_ra
        # -------------------------------
        y_reg = torch.zeros(B, C.shape[1], dtype=torch.long, device=device)

        for i in range(B):
            s = int(start_gt[i].item())
            e = int(end_gt[i].item())

            assert 0 <= s <= e < N

            y_reg[i, s:e+1] = 1

        L_ra, _, _ = ra_loss_fn(logits, y_reg, region_embed.E_reg)

        # -------------------------------
        # Total loss
        # -------------------------------
        # 🔥 Dynamic warm-up loss scheduling

        if epoch < 3:
            # Phase 1: learn localization only
            L_total = L_ml + 0.005 * L_ra

        elif epoch < 6:
            # Phase 2: introduce contrastive slowly
            L_total = L_ml + 0.005 * L_ra + 0.005 * L_sc

        else:
            # Phase 3: full training
            L_total = 1.2 * L_ml + 0.01 * L_sc + 0.005 * L_ra

        # -------------------------------
        # Backward
        # -------------------------------
        L_total.backward()
        optimizer.step()

        epoch_loss += L_total.item()

        pbar.set_postfix({
            "L_total": f"{L_total.item():.4f}",
            "L_ml": f"{L_ml.item():.4f}",
            "L_sc": f"{L_sc.item():.4f}",
            "L_ra": f"{L_ra.item():.4f}"
        })

        # -------------------------------
        # First batch debug (only once)
        # -------------------------------
        if epoch == 0 and batch_idx == 0:
            print("\n===== DEBUG (FIRST BATCH) =====")
            print("F_prime diff:", torch.norm(F_prime[0,0] - F_prime[0,1]))
            print("C_final diff:", torch.norm(C[0,0] - C[0,1]))
            print("C_prime diff:", torch.norm(C_prime[0,0] - C_prime[0,1]))
            print("Start logits sample:", start_logits[0,:5])

    # -------------------------------
    # Validation
    # -------------------------------
    r1_05, r1_07 = evaluate()

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Loss: {epoch_loss:.4f}")
    print(f"R@1 IoU@0.5: {r1_05:.4f}")
    print(f"R@1 IoU@0.7: {r1_07:.4f}")

    # -------------------------------
    # Checkpoint
    # -------------------------------
    ckpt_path = f"checkpoints/model_epoch_{epoch+1}.pth"

    torch.save({
        "epoch": epoch,
        "model_state": {m.__class__.__name__: m.state_dict() for m in modules},
        "optimizer_state": optimizer.state_dict(),
        "score": r1_05
    }, ckpt_path)

    # -------------------------------
    # Early stopping
    # -------------------------------
    if r1_05 > best_score:
        best_score = r1_05
        no_improve = 0
    else:
        no_improve += 1

    if epoch >= 5 and no_improve >= PATIENCE:
        print("\n⛔ Early stopping triggered")
        break


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])


Epoch 1/30:   0%|          | 1/1032 [00:01<30:58,  1.80s/it, L_total=3.5681, L_ml=3.5646, L_sc=5.1471, L_ra=0.6903]


===== DEBUG (FIRST BATCH) =====
F_prime diff: tensor(3.1696, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
C_final diff: tensor(1.9740, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
C_prime diff: tensor(10.3929, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Start logits sample: tensor([0.3613, 0.5886, 0.7774, 0.6797, 0.6370], device='cuda:0',
       grad_fn=<SliceBackward0>)


Epoch 1/30: 100%|██████████| 1032/1032 [33:41<00:00,  1.96s/it, L_total=3.1881, L_ml=3.1860, L_sc=5.6503, L_ra=0.4298]



Epoch 1 Summary:
Loss: 3390.1024
R@1 IoU@0.5: 0.2311
R@1 IoU@0.7: 0.1019


...


Epoch 20 Summary:
Loss: 3011.4027
R@1 IoU@0.5: 0.2481
R@1 IoU@0.7: 0.0813


Epoch 21/30: 100%|██████████| 1032/1032 [36:35<00:00,  2.13s/it, L_total=2.7388, L_ml=2.2672, L_sc=1.7004, L_ra=0.2238]



Epoch 21 Summary:
Loss: 2972.0060
R@1 IoU@0.5: 0.2612
R@1 IoU@0.7: 0.0900


Epoch 22/30: 100%|██████████| 1032/1032 [34:32<00:00,  2.01s/it, L_total=2.9306, L_ml=2.4265, L_sc=1.6843, L_ra=0.3828]


KeyboardInterrupt: 

In [7]:
# ==========================================
# 🔷 EVALUATION CALL (QCLPL NOTEBOOK)
# ==========================================

from evaluate import evaluate_model

# -------------------------------
# Provide checkpoint path
# -------------------------------
checkpoint_path = "checkpoints/model_epoch_15.pth"

# -------------------------------
# Run evaluation
# -------------------------------
results = evaluate_model(checkpoint_path, batch_size=32)

# -------------------------------
# Pretty print results
# -------------------------------
print("\n==============================")
print("    FINAL EVALUATION ")
print("==============================")

print(f"R@1 (IoU=0.5) : {results['R@1 IoU=0.5']:.4f}")
print(f"R@1 (IoU=0.7) : {results['R@1 IoU=0.7']:.4f}")

print("==============================")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 10338.29it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



 Initializing QuerySplit module...
 W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])


Evaluating: 100%|██████████| 126/126 [03:53<00:00,  1.85s/it]


    FINAL EVALUATION 
R@1 (IoU=0.5) : 2.0691
R@1 (IoU=0.7) : 0.7535


In [2]:
# ==========================================
# 🔷 FULL TRAINING PIPELINE- GPU modified
# ==========================================

import os
import torch
from tqdm import tqdm
import torch.nn.functional as F

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention
from model.bid_attn import BiDAttention
from model.region_predictor import RegionPredictor
from model.region_embedding import RegionEmbedding
from model.bigru import BiGRULocalizer

from losses.sc_loss import SemanticContrastiveLoss
from losses.ra_loss import RegionAwareLoss
from losses.pml_loss import MomentLocalizationLoss

# -------------------------------
# Device
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Hyperparameters
# -------------------------------
EPOCHS = 30
LR = 5e-5
PATIENCE = 10

W_ML = 1.0
W_SC = 0.02
W_RA = 0.005

# -------------------------------
# Initialize Model Components
# -------------------------------
query_split = QuerySplit(384).to(device)
mmt = MMT(384, 4, 1536, 1).to(device)
cross_attn = CrossAttention(384).to(device)
bid_attn = BiDAttention(384, 4).to(device)
region_pred = RegionPredictor(384).to(device)
region_embed = RegionEmbedding(384).to(device)
bigru = BiGRULocalizer().to(device)

modules = [query_split, mmt, cross_attn, bid_attn, region_pred, region_embed, bigru]

for m in modules:
    m.train()

# -------------------------------
# Losses
# -------------------------------
sc_loss_fn = SemanticContrastiveLoss()
ra_loss_fn = RegionAwareLoss()
ml_loss_fn = MomentLocalizationLoss()

# -------------------------------
# Optimizer
# -------------------------------
params = []
for m in modules:
    params += list(m.parameters())

optimizer = torch.optim.Adamax(params, lr=LR)

# -------------------------------
# Checkpoint dir
# -------------------------------
os.makedirs("checkpoints", exist_ok=True)

# -------------------------------
# Utility: IoU
# -------------------------------
def compute_iou(pred_s, pred_e, gt_s, gt_e):
    inter = max(0, min(pred_e, gt_e) - max(pred_s, gt_s) + 1)
    union = max(pred_e, gt_e) - min(pred_s, gt_s) + 1
    return inter / union if union > 0 else 0

# -------------------------------
# Validation
# -------------------------------
def evaluate():
    for m in modules:
        m.eval()

    total = 0
    correct_05 = 0
    correct_07 = 0

    with torch.no_grad():
        for batch in val_loader:

            video_feat = batch["video_feat"].to(device)
            query_feat = batch["query_feat"].to(device)
            caption_feat = batch["caption_feat"].to(device)

            start_gt = batch["start"].to(device)
            end_gt = batch["end"].to(device)

            N = video_feat.shape[1]

            Q_f, Q_s = query_split(query_feat)
            F_context = mmt(video_feat)
            S_context = mmt(caption_feat)

            F_prime = cross_attn(F_context, Q_f)
            S_prime = cross_attn(S_context, Q_s)

            C = bid_attn(F_prime, S_prime, query_feat)

            P_reg, _ = region_pred(C, use_gumbel=False)
            C_prime = region_embed(C, P_reg)

            start_logits, end_logits = bigru(C_prime)
            # 🔥 Sharpen predictions
            logit_temp = 0.3
            start_logits = start_logits / logit_temp
            end_logits   = end_logits / logit_temp
            
            start_logits = start_logits[:, :N]
            end_logits = end_logits[:, :N]

            # 🔥 TOP-K SPAN SELECTION (REPLACES ARGMAX)

            k = 3

            start_scores = start_logits
            end_scores   = end_logits

            topk_start = torch.topk(start_scores, k, dim=1)
            topk_end   = torch.topk(end_scores, k, dim=1)

            pred_s_list = topk_start.indices
            pred_e_list = topk_end.indices

            best_s = []
            best_e = []

            for i in range(start_scores.shape[0]):

                best_score = -1e9
                best_pair = (0, 0)

                for s in pred_s_list[i]:
                    for e in pred_e_list[i]:

                        s_val = s.item()
                        e_val = e.item()

                        # enforce valid span
                        if e_val < s_val:
                            continue

                        # score = start + end confidence
                        score = start_scores[i, s_val] + end_scores[i, e_val]

                        if score > best_score:
                            best_score = score
                            best_pair = (s_val, e_val)

                best_s.append(best_pair[0])
                best_e.append(best_pair[1])

            pred_s = torch.tensor(best_s, device=device)
            pred_e = torch.tensor(best_e, device=device)
            for i in range(len(pred_s)):
                iou = compute_iou(pred_s[i].item(), pred_e[i].item(),
                                  start_gt[i].item(), end_gt[i].item())

                if iou >= 0.5:
                    correct_05 += 1
                if iou >= 0.7:
                    correct_07 += 1

                total += 1

    for m in modules:
        m.train()

    return correct_05 / total, correct_07 / total

# -------------------------------
# Training Loop
# -------------------------------
best_score = 0
no_improve = 0

for epoch in range(EPOCHS):

    epoch_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch_idx, batch in enumerate(pbar):

        video_feat = batch["video_feat"].to(device)
        query_feat = batch["query_feat"].to(device)
        caption_feat = batch["caption_feat"].to(device)

        start_gt = batch["start"].to(device)
        end_gt = batch["end"].to(device)

        B, N, _ = video_feat.shape

        optimizer.zero_grad()

        # -------------------------------
        # Forward pass
        # -------------------------------
        Q_f, Q_s = query_split(query_feat)

        F_context = mmt(video_feat)
        S_context = mmt(caption_feat)

        F_prime = cross_attn(F_context, Q_f)
        S_prime = cross_attn(S_context, Q_s)

        C = bid_attn(F_prime, S_prime, query_feat)

        # Gumbel Softmax (training)
        P_reg, logits = region_pred(C, use_gumbel=True)

        C_prime = region_embed(C, P_reg)

        start_logits, end_logits = bigru(C_prime)
        # 🔥 Sharpen predictions
        logit_temp = 0.2
        start_logits = start_logits / logit_temp
        end_logits   = end_logits / logit_temp
        # -------------------------------
        # L_ml
        # -------------------------------
        L_ml, _, _ = ml_loss_fn(start_logits, end_logits, start_gt, end_gt, N)
        # -------------------------------
        # Entropy Loss (confidence boost)
        # -------------------------------
        start_prob = torch.softmax(start_logits[:, :N], dim=-1)
        end_prob   = torch.softmax(end_logits[:, :N], dim=-1)

        entropy_start = -torch.sum(start_prob * torch.log(start_prob + 1e-8), dim=1).mean()
        entropy_end   = -torch.sum(end_prob   * torch.log(end_prob   + 1e-8), dim=1).mean()

        L_entropy = entropy_start + entropy_end
        # -------------------------------
        # L_sc (negative sampling)
        # -------------------------------
       
        F_pos = F_context
        S_pos = S_context

        with torch.no_grad():

            # ---- Query representation ----
            Q_global = query_feat.mean(dim=1)                      # (B, 384)
            Q_global = F.normalize(Q_global, dim=-1)

            # ---- Feature representation (global) ----
            F_global = F_pos.mean(dim=1)                           # (B, 384)
            S_global = S_pos.mean(dim=1)

            F_global = F.normalize(F_global, dim=-1)
            S_global = F.normalize(S_global, dim=-1)

            # ---- Similarity matrix ----
            sim_f = torch.matmul(Q_global, F_global.T)             # (B, B)
            sim_s = torch.matmul(Q_global, S_global.T)

            # ---- Mask self similarity ----
            mask = torch.eye(B, device=device).bool()
            sim_f.masked_fill_(mask, -1e9)
            sim_s.masked_fill_(mask, -1e9)

            # ---- Pick hardest negatives ----
            hard_idx_f = torch.argmax(sim_f, dim=1)                # (B,)
            hard_idx_s = torch.argmax(sim_s, dim=1)

        # ---- Gather negatives ----
        F_neg = F_pos[hard_idx_f]
        S_neg = S_pos[hard_idx_s]

        # ---- Compute loss ----
        L_sc = sc_loss_fn(query_feat, F_pos, F_neg, S_pos, S_neg)

       
        # -------------------------------
        # L_ra
        # -------------------------------
        y_reg = torch.zeros(B, C.shape[1], dtype=torch.long, device=device)

        for i in range(B):
            s = int(start_gt[i].item())
            e = int(end_gt[i].item())

            assert 0 <= s <= e < N

            y_reg[i, s:e+1] = 1

        L_ra, _, _ = ra_loss_fn(logits, y_reg, region_embed.E_reg)

        # -------------------------------
        # Total loss
        # -------------------------------
        # 🔥 Dynamic warm-up loss scheduling

        if epoch < 3:
            # Phase 1: learn localization only
            L_total = L_ml + 0.005 * L_ra + 0.01 * L_entropy

        elif epoch < 6:
            # Phase 2: introduce contrastive slowly
            L_total = L_ml + 0.005 * L_ra + 0.005 * L_sc + 0.01 * L_entropy

        else:
            # Phase 3: full training
            L_total = L_ml + 0.005 * L_ra + 0.005 * L_sc + 0.01 * L_entropy

        # -------------------------------
        # Backward
        # -------------------------------
        L_total.backward()
        optimizer.step()

        epoch_loss += L_total.item()

        pbar.set_postfix({
            "L_total": f"{L_total.item():.4f}",
            "L_ml": f"{L_ml.item():.4f}",
            "L_sc": f"{L_sc.item():.4f}",
            "L_ra": f"{L_ra.item():.4f}"
        })

        # -------------------------------
        # First batch debug (only once)
        # -------------------------------
        if epoch == 0 and batch_idx == 0:
            print("\n===== DEBUG (FIRST BATCH) =====")
            print("F_prime diff:", torch.norm(F_prime[0,0] - F_prime[0,1]))
            print("C_final diff:", torch.norm(C[0,0] - C[0,1]))
            print("C_prime diff:", torch.norm(C_prime[0,0] - C_prime[0,1]))
            print("Start logits sample:", start_logits[0,:5])

    # -------------------------------
    # Validation
    # -------------------------------
    r1_05, r1_07 = evaluate()

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Loss: {epoch_loss:.4f}")
    print(f"R@1 IoU@0.5: {r1_05:.4f}")
    print(f"R@1 IoU@0.7: {r1_07:.4f}")

    # -------------------------------
    # Checkpoint
    # -------------------------------
    ckpt_path = f"checkpoints/model_epoch_{epoch+1}.pth"

    torch.save({
        "epoch": epoch,
        "model_state": {m.__class__.__name__: m.state_dict() for m in modules},
        "optimizer_state": optimizer.state_dict(),
        "score": r1_05
    }, ckpt_path)

    # -------------------------------
    # Early stopping
    # -------------------------------
    if r1_05 > best_score:
        best_score = r1_05
        no_improve = 0
    else:
        no_improve += 1

    if epoch >= 5 and no_improve >= PATIENCE:
        print("\n⛔ Early stopping triggered")
        break


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])


Epoch 1/30:   0%|          | 1/1032 [00:02<36:14,  2.11s/it, L_total=3.7316, L_ml=3.6936, L_sc=5.4225, L_ra=0.7068]


===== DEBUG (FIRST BATCH) =====
F_prime diff: tensor(3.4726, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
C_final diff: tensor(2.1092, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
C_prime diff: tensor(2.5824, device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Start logits sample: tensor([-0.6025, -0.7785, -0.7822, -0.8489, -0.9313], device='cuda:0',
       grad_fn=<SliceBackward0>)


Epoch 1/30: 100%|██████████| 1032/1032 [36:28<00:00,  2.12s/it, L_total=2.9618, L_ml=2.9270, L_sc=5.0719, L_ra=0.4690]



Epoch 1 Summary:
Loss: 3423.9979
R@1 IoU@0.5: 0.2400
R@1 IoU@0.7: 0.1084


...


Epoch 3 Summary:
Loss: 3272.4880
R@1 IoU@0.5: 0.2328
R@1 IoU@0.7: 0.1115


Epoch 4/30: 100%|██████████| 1032/1032 [34:03<00:00,  1.98s/it, L_total=3.3007, L_ml=3.2473, L_sc=4.0150, L_ra=0.3902]



Epoch 4 Summary:
Loss: 3251.5194
R@1 IoU@0.5: 0.2325
R@1 IoU@0.7: 0.1110


Epoch 5/30:   9%|▊         | 89/1032 [02:58<31:28,  2.00s/it, L_total=2.9979, L_ml=2.9411, L_sc=4.9542, L_ra=0.3061]


KeyboardInterrupt: 

In [ ]:
# ==========================================
# 🔷 EVALUATION CALL (QCLPL NOTEBOOK)
# ==========================================

from evaluate import evaluate_model

# -------------------------------
# Provide checkpoint path
# -------------------------------
checkpoint_path = "checkpoints/model_epoch_21.pth"

# -------------------------------
# Run evaluation
# -------------------------------
results = evaluate_model(checkpoint_path, batch_size=32)

# -------------------------------
# Pretty print results
# -------------------------------
print("\n==============================")
print("   🔷 FINAL EVALUATION 🔷")
print("==============================")

print(f"R@1 (IoU=0.5) : {results['R@1 IoU=0.5']:.4f}")
print(f"R@1 (IoU=0.7) : {results['R@1 IoU=0.7']:.4f}")

print("==============================")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 10943.35it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])


Evaluating: 100%|██████████| 126/126 [04:11<00:00,  2.00s/it]


   🔷 FINAL EVALUATION 🔷
R@1 (IoU=0.5) : 0.1214
R@1 (IoU=0.7) : 0.0440


: 

In [2]:
!pip install transformers

  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.1 MB 5.6 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/10.1 MB 6.4 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/10.1 MB 6.3 MB/s eta 0:00:02
   --------------- ------------------------ 3.9/10.1 MB 5.3 MB/s eta 0:00:02
   ----------------- ---------------------- 4.5/10.1 MB 4.5 MB/s eta 0:00:02
   ------------------ --------------------- 4.7/10.1 MB 4.1 MB/s eta 0:00:02
   ------------------- -------------------- 5.0/10.1 MB 3.6 MB/s eta 0:00:02
   --------------------- ------------------ 5.5/10.1 MB 3.2 MB/s eta 0:00:02
   --------------------- ------------------ 5.5/10.1 MB 3.2 MB/s eta 0:00:02
   ---------------------- ----------------- 5.8/10.1 MB 2.8 MB/s eta 0:00:02
   ---------------------- ----------------- 5.8/10.1 MB 2.8 MB/s eta 0:00:02
   ------------------

In [3]:
# ==========================================
# 🔷 QUERY SPLIT + MMT PIPELINE
# ==========================================

import torch
from tqdm import tqdm

from dataloader.query_split import QuerySplit
from model.mmt import MMT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize Modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)

query_split.train()
mmt.train()

# -------------------------------
# Loop over Dataset
# -------------------------------
for batch_idx, batch in enumerate(tqdm(train_loader, desc="Processing Dataset")):

    video_feat   = batch["video_feat"].to(device)    # (B, N, 384)
    query_feat   = batch["query_feat"].to(device)    # (B, Lq, 384)
    caption_feat = batch["caption_feat"].to(device)  # (B, Ls, 384)

    # -------------------------------
    # 1. Query Split
    # -------------------------------
    Q_f, Q_s = query_split(query_feat)               # (B, Lq, 384)

    # -------------------------------
    # 2. MMT (Context Encoding)
    # -------------------------------
    F_context = mmt(video_feat)                      # (B, N, 384)
    S_context = mmt(caption_feat)                    # (B, Ls, 384)

    # -------------------------------
    # Debug First Batch Only
    # -------------------------------
    if batch_idx == 0:
        print("\n===== SHAPE CHECK =====")
        print("Video_feat :", video_feat.shape)
        print("F_context  :", F_context.shape)

        print("Caption_feat:", caption_feat.shape)
        print("S_context   :", S_context.shape)

        print("Query_feat :", query_feat.shape)
        print("Q_f        :", Q_f.shape)
        print("Q_s        :", Q_s.shape)

    # -------------------------------
    # Next Stage (Cross Attention)
    # -------------------------------
    # Q_f ↔ F_context
    # Q_s ↔ S_context

    # Example:
    # output = model(Q_f, Q_s, F_context, S_context)


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])


Processing Dataset:   0%|          | 1/33005 [00:06<62:26:10,  6.81s/it]


===== SHAPE CHECK =====
Video_feat : torch.Size([1, 6, 384])
F_context  : torch.Size([1, 6, 384])
Caption_feat: torch.Size([1, 21, 384])
S_context   : torch.Size([1, 21, 384])
Query_feat : torch.Size([1, 8, 384])
Q_f        : torch.Size([1, 8, 384])
Q_s        : torch.Size([1, 8, 384])


Processing Dataset: 100%|██████████| 33005/33005 [2:24:05<00:00,  3.82it/s]  


In [6]:
# ==========================================
# 🔷 FORWARD PASS CHECK (ERROR-FREE TEST)
# ==========================================

import torch

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)
cross_attn = CrossAttention(dim=384).to(device)

query_split.eval()
mmt.eval()
cross_attn.eval()

# -------------------------------
# Move sample to device
# -------------------------------
video_feat   = sample["video_feat"].to(device)    # (B, N, 384)
query_feat   = sample["query_feat"].to(device)    # (B, Lq, 384)
caption_feat = sample["caption_feat"].to(device)  # (B, Ls, 384)

with torch.no_grad():

    # -------------------------------
    # 1. Query Split
    # -------------------------------
    Q_f, Q_s = query_split(query_feat)

    # -------------------------------
    # 2. MMT
    # -------------------------------
    F_context = mmt(video_feat)
    S_context = mmt(caption_feat)

    # -------------------------------
    # 3. Cross Attention (Paper)
    # -------------------------------
    F_prime = cross_attn(F_context, Q_f)
    S_prime = cross_attn(S_context, Q_s)

# -------------------------------
# Shape verification
# -------------------------------
print("\n===== FORWARD PASS SUCCESS =====")

print("Input Shapes:")
print("Video      :", video_feat.shape)
print("Query      :", query_feat.shape)
print("Caption    :", caption_feat.shape)

print("\nAfter Query Split:")
print("Q_f        :", Q_f.shape)
print("Q_s        :", Q_s.shape)

print("\nAfter MMT:")
print("F_context  :", F_context.shape)
print("S_context  :", S_context.shape)

print("\nAfter Cross Attention:")
print("F_prime    :", F_prime.shape)
print("S_prime    :", S_prime.shape)

# -------------------------------
# Assertions (strict checks)
# -------------------------------
assert Q_f.shape == query_feat.shape
assert Q_s.shape == query_feat.shape

assert F_context.shape == video_feat.shape
assert S_context.shape == caption_feat.shape

assert F_prime.shape == video_feat.shape
assert S_prime.shape == caption_feat.shape

print("\n✅ All shape checks passed. Pipeline is correct.")


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])

===== FORWARD PASS SUCCESS =====
Input Shapes:
Video      : torch.Size([1, 6, 384])
Query      : torch.Size([1, 19, 384])
Caption    : torch.Size([1, 40, 384])

After Query Split:
Q_f        : torch.Size([1, 19, 384])
Q_s        : torch.Size([1, 19, 384])

After MMT:
F_context  : torch.Size([1, 6, 384])
S_context  : torch.Size([1, 40, 384])

After Cross Attention:
F_prime    : torch.Size([1, 6, 384])
S_prime    : torch.Size([1, 40, 384])

✅ All shape checks passed. Pipeline is correct.


In [7]:
# ==========================================
# 🔷 FULL DATASET FLOW CHECK (SAFE MODE)
# ==========================================

import torch
from tqdm import tqdm

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)
cross_attn = CrossAttention(dim=384).to(device)

query_split.eval()
mmt.eval()
cross_attn.eval()

MAX_BATCHES = 100   # 🔥 change this (50–200 is ideal)

# -------------------------------
# Loop over dataset
# -------------------------------
for batch_idx, batch in enumerate(tqdm(train_loader, desc="Validating Pipeline")):

    video_feat   = batch["video_feat"].to(device)
    query_feat   = batch["query_feat"].to(device)
    caption_feat = batch["caption_feat"].to(device)

    with torch.no_grad():

        # 1. Query Split
        Q_f, Q_s = query_split(query_feat)

        # 2. MMT
        F_context = mmt(video_feat)
        S_context = mmt(caption_feat)

        # 3. Cross Attention
        F_prime = cross_attn(F_context, Q_f)
        S_prime = cross_attn(S_context, Q_s)

    # -------------------------------
    # First batch debug only
    # -------------------------------
    if batch_idx == 0:
        print("\n===== FIRST BATCH CHECK =====")
        print("Video      :", video_feat.shape)
        print("Query      :", query_feat.shape)
        print("Caption    :", caption_feat.shape)

        print("Q_f        :", Q_f.shape)
        print("F_context  :", F_context.shape)
        print("F_prime    :", F_prime.shape)

    # -------------------------------
    # Shape checks (STRICT)
    # -------------------------------
    assert Q_f.shape == query_feat.shape
    assert Q_s.shape == query_feat.shape

    assert F_context.shape == video_feat.shape
    assert S_context.shape == caption_feat.shape

    assert F_prime.shape == video_feat.shape
    assert S_prime.shape == caption_feat.shape

    # -------------------------------
    # Stop early (for speed)
    # -------------------------------
    if batch_idx >= MAX_BATCHES:
        break

print("\n✅ Pipeline verified for multiple batches — no errors.")


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])


Validating Pipeline:   0%|          | 1/33005 [00:03<35:15:27,  3.85s/it]


===== FIRST BATCH CHECK =====
Video      : torch.Size([1, 6, 384])
Query      : torch.Size([1, 12, 384])
Caption    : torch.Size([1, 32, 384])
Q_f        : torch.Size([1, 12, 384])
F_context  : torch.Size([1, 6, 384])
F_prime    : torch.Size([1, 6, 384])


Validating Pipeline:   0%|          | 100/33005 [00:25<2:17:50,  3.98it/s]


✅ Pipeline verified for multiple batches — no errors.


In [8]:
# ==========================================
# 🔷 FULL VERBOSE FORWARD PASS (1 BATCH)
# ==========================================

import torch

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention
from model.bid_attn import BiDAttention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)
cross_attn = CrossAttention(dim=384).to(device)
bid_attn = BiDAttention(dim=384, num_heads=4).to(device)

query_split.eval()
mmt.eval()
cross_attn.eval()
bid_attn.eval()

# -------------------------------
# Get one batch
# -------------------------------
batch = next(iter(train_loader))

video_feat   = batch["video_feat"].to(device)
query_feat   = batch["query_feat"].to(device)
caption_feat = batch["caption_feat"].to(device)

print("\n===== INPUT =====")
print("Video_feat   :", video_feat.shape)
print("Query_feat   :", query_feat.shape)
print("Caption_feat :", caption_feat.shape)

with torch.no_grad():

    # -------------------------------
    # 1. Query Split
    # -------------------------------
    Q_f, Q_s = query_split(query_feat)

    print("\n===== QUERY SPLIT =====")
    print("Q_f :", Q_f.shape)
    print("Q_s :", Q_s.shape)

    # -------------------------------
    # 2. MMT
    # -------------------------------
    F_context = mmt(video_feat)
    S_context = mmt(caption_feat)

    print("\n===== MMT OUTPUT =====")
    print("F_context :", F_context.shape)
    print("S_context :", S_context.shape)

    # -------------------------------
    # 3. Cross Attention (Paper)
    # -------------------------------
    F_prime = cross_attn(F_context, Q_f)
    S_prime = cross_attn(S_context, Q_s)

    print("\n===== CROSS ATTENTION =====")
    print("F_prime :", F_prime.shape)
    print("S_prime :", S_prime.shape)

    # -------------------------------
    # 4. Bi-Directional Attention
    # -------------------------------
    C_final = bid_attn(F_prime, S_prime, query_feat)

    print("\n===== BIDIRECTIONAL ATTENTION =====")
    print("C_final :", C_final.shape)

# -------------------------------
# Final Checks
# -------------------------------
assert C_final.shape[0] == video_feat.shape[0]
assert C_final.shape[2] == 384

print("\n✅ Full forward pass successful — pipeline is correct.")


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])

===== INPUT =====
Video_feat   : torch.Size([1, 6, 384])
Query_feat   : torch.Size([1, 7, 384])
Caption_feat : torch.Size([1, 34, 384])

===== QUERY SPLIT =====
Q_f : torch.Size([1, 7, 384])
Q_s : torch.Size([1, 7, 384])

===== MMT OUTPUT =====
F_context : torch.Size([1, 6, 384])
S_context : torch.Size([1, 34, 384])

===== CROSS ATTENTION =====
F_prime : torch.Size([1, 6, 384])
S_prime : torch.Size([1, 34, 384])

===== BIDIRECTIONAL ATTENTION =====
C_final : torch.Size([1, 40, 384])

✅ Full forward pass successful — pipeline is correct.


In [9]:
# ==========================================
# 🔷 FULL PIPELINE WITH DATA VISIBILITY
# ==========================================

import torch

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention
from model.bid_attn import BiDAttention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)
cross_attn = CrossAttention(dim=384).to(device)
bid_attn = BiDAttention(dim=384, num_heads=4).to(device)

query_split.eval()
mmt.eval()
cross_attn.eval()
bid_attn.eval()

# -------------------------------
# Get one batch
# -------------------------------
batch = next(iter(train_loader))

video_feat   = batch["video_feat"].to(device)
query_feat   = batch["query_feat"].to(device)
caption_feat = batch["caption_feat"].to(device)

# Helper to print small tensor sample
def print_sample(name, tensor):
    print(f"\n{name}")
    print("Shape:", tensor.shape)
    print("Sample values:", tensor[0, :2, :5])  # first 2 tokens, first 5 dims


print("\n================ INPUT =================")
print_sample("Video_feat", video_feat)
print_sample("Query_feat", query_feat)
print_sample("Caption_feat", caption_feat)

with torch.no_grad():

    # -------------------------------
    # 1. Query Split
    # -------------------------------
    Q_f, Q_s = query_split(query_feat)

    print("\n================ QUERY SPLIT =================")
    print_sample("Q_f (frame-aligned query)", Q_f)
    print_sample("Q_s (subtitle-aligned query)", Q_s)

    # -------------------------------
    # 2. MMT (Context Encoding)
    # -------------------------------
    F_context = mmt(video_feat)
    S_context = mmt(caption_feat)

    print("\n================ MMT OUTPUT =================")
    print_sample("F_context (video)", F_context)
    print_sample("S_context (subtitle)", S_context)

    # -------------------------------
    # 3. Cross Attention (Paper)
    # -------------------------------
    F_prime = cross_attn(F_context, Q_f)
    S_prime = cross_attn(S_context, Q_s)

    print("\n================ CROSS ATTENTION =================")
    print_sample("F_prime (query-conditioned video)", F_prime)
    print_sample("S_prime (query-conditioned subtitle)", S_prime)

    # -------------------------------
    # 4. Bi-Directional Attention
    # -------------------------------
    C_final = bid_attn(F_prime, S_prime, query_feat)

    print("\n================ BIDIRECTIONAL ATTENTION =================")
    print_sample("C_final (multimodal context)", C_final)

# -------------------------------
# Final Check
# -------------------------------
print("\n================ FINAL CHECK =================")
print("Final output shape:", C_final.shape)

assert C_final.shape[2] == 384

print("\n✅ Full pipeline verified with actual data flow.")


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])

================ INPUT =================

Video_feat
Shape: torch.Size([1, 6, 384])
Sample values: tensor([[ 0.3904, -0.4848,  0.0442, -1.3668,  0.5623],
        [ 0.3323, -0.6049, -0.0046, -1.2908,  0.5902]],
       grad_fn=<SliceBackward0>)

Query_feat
Shape: torch.Size([1, 14, 384])
Sample values: tensor([[-0.3239,  0.1963,  0.3126,  0.1586,  0.2660],
        [-0.5510,  0.3047,  0.4017,  0.0442,  0.1973]],
       grad_fn=<SliceBackward0>)

Caption_feat
Shape: torch.Size([1, 38, 384])
Sample values: tensor([[-0.3371,  0.1957,  0.3033,  0.1497,  0.2658],
        [-0.5834,  0.0489,  0.4262,  0.1465,  0.1625]],
       grad_fn=<SliceBackward0>)

================ QUERY SPLIT =================

Q_f (frame-aligned query)
Shape: torch.Size([1, 14, 384])
Sample values: tensor([[ 0.1221,  0.0238, -0.0288,  0.0997,  0.0868],
        [ 0.0719, -0.063

In [2]:
# ==========================================
# 🔷 FULL PIPELINE WITH FIXED BiDAtt
# ==========================================

import torch

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention
from model.bid_attn import BiDAttention

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)
cross_attn = CrossAttention(dim=384).to(device)
bid_attn = BiDAttention(dim=384, num_heads=4).to(device)

query_split.eval()
mmt.eval()
cross_attn.eval()
bid_attn.eval()

# -------------------------------
# Get one batch
# -------------------------------
batch = next(iter(train_loader))

video_feat   = batch["video_feat"].to(device)
query_feat   = batch["query_feat"].to(device)
caption_feat = batch["caption_feat"].to(device)

print("\n===== INPUT =====")
print("Video_feat   :", video_feat.shape)
print("Query_feat   :", query_feat.shape)
print("Caption_feat :", caption_feat.shape)

with torch.no_grad():

    # 1. Query Split
    Q_f, Q_s = query_split(query_feat)

    print("\n===== QUERY SPLIT =====")
    print("Q_f :", Q_f.shape)
    print("Q_s :", Q_s.shape)

    # 2. MMT
    F_context = mmt(video_feat)
    S_context = mmt(caption_feat)

    print("\n===== MMT =====")
    print("F_context :", F_context.shape)
    print("S_context :", S_context.shape)

    # 3. Cross Attention
    F_prime = cross_attn(F_context, Q_f)
    S_prime = cross_attn(S_context, Q_s)

    print("\n===== CROSS ATTENTION =====")
    print("F_prime :", F_prime.shape)
    print("S_prime :", S_prime.shape)

    # 4. BiDAtt (FIXED)
    C_final = bid_attn(F_prime, S_prime, query_feat)

    print("\n===== BIDIRECTIONAL ATTENTION (FIXED) =====")
    print("C_final :", C_final.shape)

# -------------------------------
# Final validation
# -------------------------------
assert C_final.shape[1] == F_prime.shape[1] + S_prime.shape[1]
assert C_final.shape[2] == 384

print("\n✅ Fixed BiDAtt working — no sequence collapse.")


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])

===== INPUT =====
Video_feat   : torch.Size([1, 6, 384])
Query_feat   : torch.Size([1, 7, 384])
Caption_feat : torch.Size([1, 21, 384])

===== QUERY SPLIT =====
Q_f : torch.Size([1, 7, 384])
Q_s : torch.Size([1, 7, 384])

===== MMT =====
F_context : torch.Size([1, 6, 384])
S_context : torch.Size([1, 21, 384])

===== CROSS ATTENTION =====
F_prime : torch.Size([1, 6, 384])
S_prime : torch.Size([1, 21, 384])

===== BIDIRECTIONAL ATTENTION (FIXED) =====
C_final : torch.Size([1, 27, 384])

✅ Fixed BiDAtt working — no sequence collapse.


In [18]:
print(C_final[0, 0, :5])
print(C_final[0, 10, :5])
print(C_final[0, 30, :5])

tensor([-0.0805, -0.0307,  0.0116,  0.0089,  0.0215])
tensor([-0.0804, -0.0307,  0.0116,  0.0089,  0.0217])
tensor([-0.0804, -0.0307,  0.0116,  0.0089,  0.0217])


In [19]:
print("F_prime diff:", torch.norm(F_prime[0,0] - F_prime[0,1]))
print("S_prime diff:", torch.norm(S_prime[0,0] - S_prime[0,1]))

F_prime diff: tensor(2.0895)
S_prime diff: tensor(2.4670)


In [20]:
print("F_context diff:", torch.norm(F_context[0,0] - F_context[0,1]))
print("S_context diff:", torch.norm(S_context[0,0] - S_context[0,1]))

F_context diff: tensor(9.0847)
S_context diff: tensor(11.4928)


In [21]:
print("video_feat diff:", torch.norm(video_feat[0,0] - video_feat[0,1]))

video_feat diff: tensor(7.1619, grad_fn=<LinalgVectorNormBackward0>)


In [3]:
print("\n===== COLLAPSE CHECK =====")

print("Video raw diff:",
      torch.norm(video_feat[0,0] - video_feat[0,1]))

print("After MMT diff:",
      torch.norm(F_context[0,0] - F_context[0,1]))

print("After CrossAttn diff:",
      torch.norm(F_prime[0,0] - F_prime[0,1]))

print("Final diff:",
      torch.norm(C_final[0,0] - C_final[0,1]))


===== COLLAPSE CHECK =====
Video raw diff: tensor(5.3313, grad_fn=<LinalgVectorNormBackward0>)
After MMT diff: tensor(6.7260)
After CrossAttn diff: tensor(2.3028)
Final diff: tensor(1.3433)


In [4]:
# ==========================================
# 🔷 FULL PIPELINE: END-TO-END FORWARD PASS
# ==========================================

import torch

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention
from model.bid_attn import BiDAttention
from model.region_predictor import RegionPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)
cross_attn = CrossAttention(dim=384).to(device)
bid_attn = BiDAttention(dim=384, num_heads=4).to(device)
region_pred = RegionPredictor(dim=384).to(device)

query_split.eval()
mmt.eval()
cross_attn.eval()
bid_attn.eval()
region_pred.eval()

# -------------------------------
# Get one batch
# -------------------------------
batch = next(iter(train_loader))

video_feat   = batch["video_feat"].to(device)
query_feat   = batch["query_feat"].to(device)
caption_feat = batch["caption_feat"].to(device)

print("\n===== INPUT =====")
print("Video_feat   :", video_feat.shape)
print("Query_feat   :", query_feat.shape)
print("Caption_feat :", caption_feat.shape)

with torch.no_grad():

    # -------------------------------
    # 1. Query Split
    # -------------------------------
    Q_f, Q_s = query_split(query_feat)

    print("\n===== QUERY SPLIT =====")
    print("Q_f :", Q_f.shape)
    print("Q_s :", Q_s.shape)

    # -------------------------------
    # 2. MMT
    # -------------------------------
    F_context = mmt(video_feat)
    S_context = mmt(caption_feat)

    print("\n===== MMT =====")
    print("F_context :", F_context.shape)
    print("S_context :", S_context.shape)

    # -------------------------------
    # 3. Cross Attention
    # -------------------------------
    F_prime = cross_attn(F_context, Q_f)
    S_prime = cross_attn(S_context, Q_s)

    print("\n===== CROSS ATTENTION =====")
    print("F_prime :", F_prime.shape)
    print("S_prime :", S_prime.shape)

    # -------------------------------
    # 4. BiDAtt
    # -------------------------------
    C_final = bid_attn(F_prime, S_prime, query_feat)

    print("\n===== BIDIRECTIONAL ATTENTION =====")
    print("C_final :", C_final.shape)

    # -------------------------------
    # 5. Region Predictor
    # -------------------------------
    P_reg, logits = region_pred(C_final, use_gumbel=False)

    print("\n===== REGION PREDICTOR =====")
    print("P_reg   :", P_reg.shape)
    print("Logits  :", logits.shape)

    # -------------------------------
    # Check probability validity
    # -------------------------------
    print("\nSample probabilities (first 5 positions):")
    print(P_reg[0, :5])

    print("\nSum across classes (should be 1):")
    print(P_reg[0, :5].sum(dim=-1))

# -------------------------------
# Final Checks
# -------------------------------
assert P_reg.shape[-1] == 2
assert C_final.shape[1] == P_reg.shape[1]

print("\n✅ Full pipeline including Region Predictor is correct.")


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])

===== INPUT =====
Video_feat   : torch.Size([1, 6, 384])
Query_feat   : torch.Size([1, 9, 384])
Caption_feat : torch.Size([1, 37, 384])

===== QUERY SPLIT =====
Q_f : torch.Size([1, 9, 384])
Q_s : torch.Size([1, 9, 384])

===== MMT =====
F_context : torch.Size([1, 6, 384])
S_context : torch.Size([1, 37, 384])

===== CROSS ATTENTION =====
F_prime : torch.Size([1, 6, 384])
S_prime : torch.Size([1, 37, 384])

===== BIDIRECTIONAL ATTENTION =====
C_final : torch.Size([1, 43, 384])

===== REGION PREDICTOR =====
P_reg   : torch.Size([1, 43, 2])
Logits  : torch.Size([1, 43, 2])

Sample probabilities (first 5 positions):
tensor([[0.5172, 0.4828],
        [0.5145, 0.4855],
        [0.5074, 0.4926],
        [0.5095, 0.4905],
        [0.5027, 0.4973]])

Sum across classes (should be 1):
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

✅ Full pipeline 

In [5]:
# ==========================================
# 🔷 FULL PIPELINE: UPTO REGION EMBEDDING
# ==========================================

import torch

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention
from model.bid_attn import BiDAttention
from model.region_predictor import RegionPredictor
from model.region_embedding import RegionEmbedding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)
cross_attn = CrossAttention(dim=384).to(device)
bid_attn = BiDAttention(dim=384, num_heads=4).to(device)
region_pred = RegionPredictor(dim=384).to(device)
region_embed = RegionEmbedding(dim=384).to(device)

query_split.eval()
mmt.eval()
cross_attn.eval()
bid_attn.eval()
region_pred.eval()
region_embed.eval()

# -------------------------------
# Get one batch
# -------------------------------
batch = next(iter(train_loader))

video_feat   = batch["video_feat"].to(device)
query_feat   = batch["query_feat"].to(device)
caption_feat = batch["caption_feat"].to(device)

print("\n===== INPUT =====")
print("Video_feat   :", video_feat.shape)
print("Query_feat   :", query_feat.shape)
print("Caption_feat :", caption_feat.shape)

with torch.no_grad():

    # -------------------------------
    # 1. Query Split
    # -------------------------------
    Q_f, Q_s = query_split(query_feat)

    print("\n===== QUERY SPLIT =====")
    print("Q_f :", Q_f.shape)
    print("Q_s :", Q_s.shape)

    # -------------------------------
    # 2. MMT
    # -------------------------------
    F_context = mmt(video_feat)
    S_context = mmt(caption_feat)

    print("\n===== MMT =====")
    print("F_context :", F_context.shape)
    print("S_context :", S_context.shape)

    # -------------------------------
    # 3. Cross Attention
    # -------------------------------
    F_prime = cross_attn(F_context, Q_f)
    S_prime = cross_attn(S_context, Q_s)

    print("\n===== CROSS ATTENTION =====")
    print("F_prime :", F_prime.shape)
    print("S_prime :", S_prime.shape)

    # -------------------------------
    # 4. BiDAtt
    # -------------------------------
    C_final = bid_attn(F_prime, S_prime, query_feat)

    print("\n===== BIDIRECTIONAL ATTENTION =====")
    print("C_final :", C_final.shape)

    # -------------------------------
    # 5. Region Predictor
    # -------------------------------
    P_reg, logits = region_pred(C_final, use_gumbel=False)

    print("\n===== REGION PREDICTOR =====")
    print("P_reg :", P_reg.shape)

    # -------------------------------
    # 6. Region Embedding
    # -------------------------------
    C_prime = region_embed(C_final, P_reg)

    print("\n===== REGION EMBEDDING =====")
    print("C_prime :", C_prime.shape)

    # -------------------------------
    # Inspect values
    # -------------------------------
    print("\nSample P_reg (first 3 positions):")
    print(P_reg[0, :3])

    print("\nSample C_final vs C_prime difference:")
    print((C_prime - C_final)[0, :3, :5])

# -------------------------------
# Final checks
# -------------------------------
assert C_prime.shape == C_final.shape
assert C_prime.shape[2] == 384

print("\n✅ Full pipeline upto Region Embedding is correct.")


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])

===== INPUT =====
Video_feat   : torch.Size([1, 6, 384])
Query_feat   : torch.Size([1, 9, 384])
Caption_feat : torch.Size([1, 24, 384])

===== QUERY SPLIT =====
Q_f : torch.Size([1, 9, 384])
Q_s : torch.Size([1, 9, 384])

===== MMT =====
F_context : torch.Size([1, 6, 384])
S_context : torch.Size([1, 24, 384])

===== CROSS ATTENTION =====
F_prime : torch.Size([1, 6, 384])
S_prime : torch.Size([1, 24, 384])

===== BIDIRECTIONAL ATTENTION =====
C_final : torch.Size([1, 30, 384])

===== REGION PREDICTOR =====
P_reg : torch.Size([1, 30, 2])

===== REGION EMBEDDING =====
C_prime : torch.Size([1, 30, 384])

Sample P_reg (first 3 positions):
tensor([[0.4903, 0.5097],
        [0.4995, 0.5005],
        [0.4873, 0.5127]])

Sample C_final vs C_prime difference:
tensor([[ 0.5017,  1.1375, -0.2422, -0.5574,  0.0777],
        [ 0.4970,  1.1549, -0.2525, -

In [6]:
# ==========================================
# 🔷 FULL END-TO-END PIPELINE (FINAL)
# ==========================================

import torch

from dataloader.query_split import QuerySplit
from model.mmt import MMT
from model.cross_attn import CrossAttention
from model.bid_attn import BiDAttention
from model.region_predictor import RegionPredictor
from model.region_embedding import RegionEmbedding
from model.bigru import BiGRULocalizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------
# Initialize modules
# -------------------------------
query_split = QuerySplit(dim=384).to(device)
mmt = MMT(dim=384, num_heads=4, ff_dim=1536, num_layers=1).to(device)
cross_attn = CrossAttention(dim=384).to(device)
bid_attn = BiDAttention(dim=384, num_heads=4).to(device)
region_pred = RegionPredictor(dim=384).to(device)
region_embed = RegionEmbedding(dim=384).to(device)
bigru = BiGRULocalizer().to(device)

# eval mode (just forward check)
query_split.eval()
mmt.eval()
cross_attn.eval()
bid_attn.eval()
region_pred.eval()
region_embed.eval()
bigru.eval()

# -------------------------------
# Get one batch
# -------------------------------
batch = next(iter(train_loader))

video_feat   = batch["video_feat"].to(device)
query_feat   = batch["query_feat"].to(device)
caption_feat = batch["caption_feat"].to(device)

print("\n===== INPUT =====")
print("Video_feat   :", video_feat.shape)
print("Query_feat   :", query_feat.shape)
print("Caption_feat :", caption_feat.shape)

with torch.no_grad():

    # -------------------------------
    # 1. Query Split
    # -------------------------------
    Q_f, Q_s = query_split(query_feat)

    print("\n===== QUERY SPLIT =====")
    print("Q_f :", Q_f.shape)
    print("Q_s :", Q_s.shape)

    # -------------------------------
    # 2. MMT
    # -------------------------------
    F_context = mmt(video_feat)
    S_context = mmt(caption_feat)

    print("\n===== MMT =====")
    print("F_context :", F_context.shape)
    print("S_context :", S_context.shape)

    # -------------------------------
    # 3. Cross Attention
    # -------------------------------
    F_prime = cross_attn(F_context, Q_f)
    S_prime = cross_attn(S_context, Q_s)

    print("\n===== CROSS ATTENTION =====")
    print("F_prime :", F_prime.shape)
    print("S_prime :", S_prime.shape)

    # -------------------------------
    # 4. BiDAtt
    # -------------------------------
    C_final = bid_attn(F_prime, S_prime, query_feat)

    print("\n===== BIDIRECTIONAL ATTENTION =====")
    print("C_final :", C_final.shape)

    # -------------------------------
    # 5. Region Predictor
    # -------------------------------
    P_reg, logits = region_pred(C_final)

    print("\n===== REGION PREDICTOR =====")
    print("P_reg :", P_reg.shape)

    # -------------------------------
    # 6. Region Embedding
    # -------------------------------
    C_prime = region_embed(C_final, P_reg)

    print("\n===== REGION EMBEDDING =====")
    print("C_prime :", C_prime.shape)

    # -------------------------------
    # 7. BiGRU + Localization
    # -------------------------------
    start_logits, end_logits = bigru(C_prime)

    print("\n===== TEMPORAL LOCALIZATION =====")
    print("Start logits :", start_logits.shape)
    print("End logits   :", end_logits.shape)

    # -------------------------------
    # Sample outputs
    # -------------------------------
    print("\nSample start logits (first 5):")
    print(start_logits[0, :5])

    print("\nSample end logits (first 5):")
    print(end_logits[0, :5])

# -------------------------------
# Final Checks
# -------------------------------
assert start_logits.shape == end_logits.shape
assert start_logits.shape[1] == C_prime.shape[1]

print("\n✅ FULL MODEL PIPELINE COMPLETE AND CORRECT.")


🚀 Initializing QuerySplit module...
✅ W_f and W_s initialized (Linear layers)
W_f shape: torch.Size([384, 384])
W_s shape: torch.Size([384, 384])

===== INPUT =====
Video_feat   : torch.Size([1, 6, 384])
Query_feat   : torch.Size([1, 6, 384])
Caption_feat : torch.Size([1, 48, 384])

===== QUERY SPLIT =====
Q_f : torch.Size([1, 6, 384])
Q_s : torch.Size([1, 6, 384])

===== MMT =====
F_context : torch.Size([1, 6, 384])
S_context : torch.Size([1, 48, 384])

===== CROSS ATTENTION =====
F_prime : torch.Size([1, 6, 384])
S_prime : torch.Size([1, 48, 384])

===== BIDIRECTIONAL ATTENTION =====
C_final : torch.Size([1, 54, 384])

===== REGION PREDICTOR =====
P_reg : torch.Size([1, 54, 2])

===== REGION EMBEDDING =====
C_prime : torch.Size([1, 54, 384])

===== TEMPORAL LOCALIZATION =====
Start logits : torch.Size([1, 54])
End logits   : torch.Size([1, 54])

Sample start logits (first 5):
tensor([-0.1111, -0.0965, -0.0814, -0.0650, -0.0721])

Sample end logits (first 5):
tensor([-0.0194,  0.0364